# 01 — Prompt Engineering Basics

Companion notebook to `01-llm-fundamentals-and-prompt-engineering.md`.

This notebook runs against a **real local LLM** via [Ollama](https://ollama.com) — no cloud API key
required, no per-call cost. We use the `llama3.2` model running on your machine to see how
zero-shot, few-shot, and chain-of-thought prompting each change the model's actual behavior.

> **Prerequisites**
> - Ollama installed and running (`ollama serve`, or the Ollama desktop app)
> - The model pulled locally: `ollama pull llama3.2`
>
> To swap in a different local model, change `OLLAMA_MODEL` below. To swap to Azure OpenAI in
> production, see the final section of this notebook.

In [ ]:
import requests

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2"

try:
    tags = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5).json()
    available = [m["name"] for m in tags.get("models", [])]
    print(f"Ollama is reachable at {OLLAMA_BASE_URL}")
    print("Available models:", available)
    if not any(OLLAMA_MODEL in m for m in available):
        print(f"\nWarning: '{OLLAMA_MODEL}' not found. Pull it with: ollama pull {OLLAMA_MODEL}")
except requests.exceptions.ConnectionError:
    print(
        f"Could not reach Ollama at {OLLAMA_BASE_URL}.\n"
        "Start it with `ollama serve` (or launch the Ollama desktop app), then re-run this cell."
    )

## A local LLM via Ollama

Unlike a mocked/canned response, `ollama_call()` below sends the prompt to the real `llama3.2`
model over Ollama's local REST API (`POST /api/generate`) and returns its actual generated text.
Because it's a real model, outputs are non-deterministic — re-running a cell may give a slightly
different answer each time, and the model may not always match the *exact* format we ask for.
That's the honest picture of how prompting actually works, versus a canned demo.

In [ ]:
def ollama_call(prompt: str, model: str = OLLAMA_MODEL, temperature: float = 0.2) -> str:
    """Send a prompt to a local Ollama model and return its generated text."""
    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": temperature},
            },
            timeout=120,
        )
        response.raise_for_status()
    except requests.exceptions.ConnectionError as exc:
        raise RuntimeError(
            f"Could not reach Ollama at {OLLAMA_BASE_URL}. Is it running? "
            "Start it with `ollama serve`."
        ) from exc
    return response.json()["response"].strip()

## 1. Zero-shot prompting

No examples, just an instruction and the question. Fast to write, and works well when the task is
common enough that the base model has seen many similar completions during training.

In [ ]:
zero_shot_prompt = """You are an internal assistant for Acme Bank.
Answer the user's question concisely.

Question: How long do refunds take?
"""

print(ollama_call(zero_shot_prompt))

## 2. Few-shot prompting

We show the model 1-2 example input/output pairs *in the prompt itself* before the real question.
This is the go-to technique when you need a specific output **format**, not just a correct answer —
exactly the pattern a production chatbot uses to force a consistent `Answer: ... / Source: ...`
shape the UI can reliably render.

In [ ]:
few_shot_prompt = """You are an internal assistant for Acme Bank.
Answer using this exact format, based on the example below.

Example:
Question: What are your branch hours?
Answer: Branches are open 9am-5pm, Monday to Friday.
Source: Branch Operations Handbook

Question: How long do refunds take?
"""

print(ollama_call(few_shot_prompt))

Notice the model tends to follow the `Answer:` / `Source:` shape here — the prompt itself
*taught* it that shape, via the example. This is next-token prediction in action: the model
continues whatever pattern the few-shot examples established in the context window (see Chapter 1).
With a small local model like `llama3.2` (3B parameters), the format may not be followed as
reliably as it would be with a larger frontier model — that gap is itself a useful, honest data
point about model capacity versus prompt technique.

## 3. Chain-of-thought (CoT) prompting

Asking the model to reason step-by-step before giving a final answer. This matters most for
multi-step reasoning tasks — like the math word problems in the Text-to-Math agent case study
(Chapter 5) — because it gives the model more forward passes (more tokens) to work through
intermediate steps before committing to a final answer.

In [ ]:
cot_prompt = """Solve the following problem. Let's think step by step, then give a Final Answer.

Question: A train travels 60 miles in 45 minutes. What is its speed in mph?
"""

print(ollama_call(cot_prompt))

## 4. Optional: the same thing via LangChain's `ChatOllama`

If `langchain_ollama` and `langchain_core` are installed, we can wire the exact same prompt
*structures* through a real LangChain `PromptTemplate` + `ChatOllama` — the same `Runnable`
interface you'd actually use in the chatbot codebase (Chapter 2), just pointed at your local model
instead of Azure OpenAI. If the packages aren't installed, this cell degrades gracefully.

Install with: `pip install langchain-core langchain-ollama`

In [ ]:
try:
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import PromptTemplate

    llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.2)

    template = PromptTemplate.from_template(
        "You are an internal assistant for {client_name}.\n"
        "Question: {question}\n"
    )

    chain = template | llm  # this is LCEL -- see notebook 02 for much more
    result = chain.invoke({"client_name": "Acme Bank", "question": "How long do refunds take?"})
    print(result.content)
except ImportError:
    print("langchain_ollama / langchain_core not installed -- install with "
          "`pip install langchain-core langchain-ollama` to run this cell.\n"
          "The ollama_call() demos above already show the core concepts without these dependencies.")

## Swapping models

**Different local model:** change `OLLAMA_MODEL` at the top of this notebook to any model you've
pulled (`ollama pull <name>`, then `ollama list` to see what is available).

**Production (Azure OpenAI):** swap the call itself, keeping the same prompt strings built above:

```python
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_deployment="<your-deployment-name>",
    api_version="2024-05-01-preview",
    temperature=0.2,
)
# requires AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT env vars set
response = llm.invoke(few_shot_prompt)
print(response.content)
```

See `03-chatbot-architecture-azure-openai.md` for how deployments and rate limits work in Azure OpenAI.